# Exploratory data analysis: CelebA-Spoof

Thin wrapper over `checkpoint1.eda`. Label statistics use every image; pixel statistics use a fixed-seed sample stratified by spoof type.

**Data root:** `CELEBA_SPOOF_ROOT` if set, else `data/CelebA_Spoof`, else the synthetic fixture in `data/fixture/CelebA_Spoof` (create it with `python -m checkpoint1.fixture`). The fixture run uses 100 images and writes to `outputs/fixture/` so it never overwrites real results. `EDA_N_SAMPLE`, `EDA_OUT_DIR` and `EDA_FIG_DIR` override the defaults.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from checkpoint1 import data, eda

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()


def _abs(p):
    p = Path(p)
    return p if p.is_absolute() else REPO / p


# data root: $CELEBA_SPOOF_ROOT, then the real dataset, then the synthetic fixture
use_fixture = False
if 'CELEBA_SPOOF_ROOT' in os.environ:
    root = _abs(os.environ['CELEBA_SPOOF_ROOT'])
elif not data.missing_parts(_abs(data.DEFAULT_ROOT)):
    root = _abs(data.DEFAULT_ROOT)
elif not data.missing_parts(_abs(data.FIXTURE_ROOT)):
    root, use_fixture = _abs(data.FIXTURE_ROOT), True
else:
    root = _abs(data.DEFAULT_ROOT)  # run_eda exits with a readable "dataset not found" message

sub = 'outputs/fixture' if use_fixture else 'outputs'
out_dir = _abs(os.environ.get('EDA_OUT_DIR', f'{sub}/eda'))
fig_dir = _abs(os.environ.get('EDA_FIG_DIR', f'{sub}/figures'))
n_sample = int(os.environ.get('EDA_N_SAMPLE', 100 if use_fixture else 10_000))
if use_fixture:
    print(f'Using synthetic fixture at {root} (real dataset not found). Figures go to {fig_dir}.')

labels, df, report = eda.run_eda(root, n_sample, data.SEED, out_dir, fig_dir)
pd.read_csv(out_dir / 'summary.csv')

In [ ]:
def show(*idx):
    for i in idx:
        name, title, caption = report.items[i - 1]
        display(Image(filename=fig_dir / name, width=700))
        display(Markdown(f'**Figure {i}: {title}.** {caption}'))

## Class balance

In [ ]:
show(1, 2, 3, 4)

## Univariate analysis

In [ ]:
show(5, 6, 7, 8)

## Bivariate analysis

In [ ]:
show(9, 10, 11, 12)

## Qualitative samples

In [ ]:
show(13)

## What this means for the project

- **Imbalance** (figures 1-3): report per-class error rates (APCER/BPCER) and per attack type; use class-weighted loss or balanced sampling.
- **Subjects** (figure 4): keep each subject in one partition; the official train/test split is subject-disjoint (see `subjects_in_both_splits` in the summary).
- **Resolution and box size** (figures 5, 8): resize inputs to one shape and check that raw resolution does not leak the label.
- **Face vs background** (figures 9-10): the more the two views differ between live and spoof, the stronger the case for separate face and environment experts.
- **Conditions and attributes** (figures 11-12): illumination/environment are spoof-only labels and may be confounded with attack type; large attribute correlations flag shortcut or bias risks.
- **Anomalies**: the `anomalies` and `sample` rows of the summary table feed the schema-sanity paragraph of the report.